# ClusterBP v2: сдвиг во времени и качество по подгруппам

**Часть 1. Сдвиг доли событий B.** В тестовом периоде временной валидации доля окон с B скачет по месяцам
(17% → 11% → 22%, `temporal_monthly.csv`), а для A стабильна. Здесь:
- какой из кодов B (33601, 33801, 33802, 33810, 33911) даёт скачки. Если один код — это, скорее всего, изменение
  настройки системы мониторинга, а не физиология; причину нужно подтвердить у оператора;
- помогают ли эксплуатационные меры: **перекалибровка** на последнем месяце с известными исходами
  (только сдвиг и сдвиг + наклон) и **ежемесячное дообучение** на всех окнах с известными исходами.
  Исход окна известен через 14 дней после отсечки, поэтому для месяца M используются окна с `cut + 14 ≤ начало M`.

**Часть 2. Подгруппы.** Качество XGB_full (out-of-fold из кросс-валидации) по возрасту, ИМТ, ветви программы,
интенсивности измерений, длительности наблюдения, фенотипу и региону. Пола в выгрузке оператора нет, поэтому
анализ по полу невозможен — это стоит указать в ограничениях. Порог — тот же, что в кросс-валидации (с внутренней валидации),
поэтому различия чувствительности и доли ложных тревог между группами — это то, что увидел бы врач.

In [ ]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LogisticRegression

from clusterbp_lib import (RESULTS_DIR, DATA_DIR, CACHE_DIR, MODELS_DIR, SEED, TARGETS, FEATURE_SETS, N_FOLDS, fold_masks,
                           fit_phenotypes, assign_phenotypes, set_phenotype_columns, best_threshold, logit, load_regions)
from clusterbp_extra import (load_windows, load_oof_scores, phenotypes, event_target, fit_predict, point_metrics,
                             paired_boot, ci, calib)

warnings.filterwarnings('ignore')
OUT = os.path.join(RESULTS_DIR, 'shift_subgroups')
os.makedirs(OUT, exist_ok=True)
FULL = FEATURE_SETS['full']
S_TARGETS = ['A', 'B']

win, pheno_win, pheno_first, pheno_first_pid = load_windows()
meas = pd.read_parquet(os.path.join(DATA_DIR, 'measurements.parquet'), columns=['pid', 'td'])
events = pd.read_parquet(os.path.join(DATA_DIR, 'events.parquet'))
pid, cut = win['pid'].values, win['cut'].values
month = pd.to_datetime(cut * 86400, unit='s').to_period('M').astype(str)

## 1.1. Какие коды B дают скачки

Слева — доля окон, в горизонте которых есть событие данного кода (как цель B: зазор 7 дней, горизонт 7 дней).
Справа — число событий кода на 100 пациентов, у которых в этом месяце были измерения.

In [ ]:
B_CODES = TARGETS['B']['codes']
code_share = pd.DataFrame({'месяц': month})
for c in B_CODES:
    code_share[str(c)] = event_target(win, events, [c], TARGETS['B']['gap'], TARGETS['B']['horizon'])
code_share['B (любой)'] = win['y_B'].values
code_share['A (для сравнения)'] = win['y_A'].values
by_month = code_share.groupby('месяц').mean()
by_month['окон'] = code_share.groupby('месяц').size()
by_month = by_month[by_month['окон'] >= 100]          # неполные месяцы в начале и конце выгрузки

ev = events.copy()
ev['месяц'] = pd.to_datetime(ev['td'] * 86400, unit='s').dt.to_period('M').astype(str)
active = meas.assign(месяц=pd.to_datetime(meas['td'] * 86400, unit='s').dt.to_period('M').astype(str)) \
    .groupby('месяц')['pid'].nunique()
rate = ev[ev.code.isin(B_CODES)].groupby(['месяц', 'code']).size().unstack(fill_value=0)
rate = rate.div(active.reindex(rate.index), axis=0) * 100
rate['активных пациентов'] = active.reindex(rate.index)
rate = rate[rate.index.isin(by_month.index)]
by_month.to_csv(os.path.join(OUT, 'B_codes_window_share_by_month.csv'))
rate.to_csv(os.path.join(OUT, 'B_codes_per_100_patients_by_month.csv'))
display(by_month.round(3))
display(rate.round(1))

# вклад каждого кода в изменение доли B между соседними месяцами (по долям окон с кодом)
diff = by_month[[str(c) for c in B_CODES] + ['B (любой)']].diff().dropna()
diff.to_csv(os.path.join(OUT, 'B_codes_month_to_month_change.csv'))
display(diff.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for c in [str(c) for c in B_CODES] + ['B (любой)']:
    axes[0].plot(by_month.index, by_month[c], marker='o', ms=3, label=c, lw=2.5 if c == 'B (любой)' else 1.2)
axes[0].set_ylabel('доля окон с событием в горизонте')
axes[0].legend(fontsize=8)
for c in B_CODES:
    if c in rate:
        axes[1].plot(rate.index, rate[c], marker='o', ms=3, label=str(c))
axes[1].set_ylabel('событий на 100 активных пациентов')
axes[1].legend(fontsize=8)
for ax in axes:
    ax.tick_params(axis='x', rotation=60)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'B_codes_by_month.png'), dpi=200, bbox_inches='tight')
plt.show()

## 1.2. Статическая модель, перекалибровка и ежемесячное дообучение

Разбиение — как в `clusterbp_v2_temporal.ipynb`: обучение — окна с `cut + 14 ≤ 01.10.2024`, внутренняя валидация —
последние 30 дней обучающего периода, тест — окна с отсечкой с 01.10.2024. Шаблоны фенотипов — по пациентам,
первые 30 дней которых целиком пришлись на обучающий период (при дообучении шаблоны не меняются).

Для каждого тестового месяца M:
- **статическая** модель — обучена один раз, порог с валидации;
- **перекалибровка (сдвиг)** — logit p' = logit p + a, a подбирается на окнах последнего месяца с известными исходами;
- **перекалибровка (сдвиг и наклон)** — logit p' = a + b · logit p на тех же окнах;
  у обеих перекалибровок порог Macro F1 подбирается заново на тех же окнах;
- **дообучение** — XGB_full заново на всех окнах с известными исходами (ранняя остановка и порог — на последних 30 днях).

In [ ]:
SPLIT = pd.Timestamp('2024-10-01')
LAG = max(TARGETS[t]['gap'] + TARGETS[t]['horizon'] for t in S_TARGETS)
VAL_DAYS = 30
T = SPLIT.value / 86400e9
train_all = cut + LAG <= T
va0 = train_all & (cut > T - LAG - VAL_DAYS)
tr0 = train_all & ~va0
te = cut >= T

tpath = os.path.join(MODELS_DIR, 'templates_temporal.npy')
if os.path.exists(tpath):
    templates = np.load(tpath)
else:
    first_day = meas.groupby('pid')['td'].min().apply(np.floor)
    eligible = first_day.index.values[(first_day.values + 30 <= T - LAG)]
    templates, _, _ = fit_phenotypes(pheno_first, pheno_first_pid, eligible, SEED)
    np.save(tpath, templates)
cache = os.path.join(CACHE_DIR, 'pheno_assign_temporal.npz')
if os.path.exists(cache):
    z = np.load(cache)
    cl, dist = z['cl'], z['dist']
else:
    cl, dist = assign_phenotypes(pheno_win, templates)
    np.savez(cache, cl=cl, dist=dist)
set_phenotype_columns(win, cl, dist)
X = win[FULL].values.astype(float)

test_months = sorted(np.unique(month[te]))
rows, monthly_preds = [], {}
t0 = time.time()
for t in S_TARGETS:
    y = win[f'y_{t}'].values
    s_static, thr_static, _ = fit_predict(win, FULL, y, tr0, va0, SEED, X=X)
    for mo in test_months:
        M = pd.Timestamp(mo + '-01').value / 86400e9
        m_te = te & (month == mo)
        if y[m_te].sum() < 20:
            continue
        known = cut + LAG <= M
        recent = known & (cut > M - LAG - 30)           # последний месяц с известными исходами
        variants = {}
        variants['статическая'] = (s_static, thr_static)
        z = logit(s_static)
        # сдвиг: a при фиксированном наклоне 1 — одномерная минимизация логистической потери
        zr, yr = z[recent], y[recent]
        nll = lambda a: np.mean(np.logaddexp(0, zr + a) - yr * (zr + a))
        a = minimize_scalar(nll, bounds=(-5, 5), method='bounded').x
        s_shift = 1 / (1 + np.exp(-(z + a)))
        variants['перекалибровка: сдвиг'] = (s_shift, best_threshold(yr, s_shift[recent])[0])
        lr = LogisticRegression(C=1e6, max_iter=1000).fit(zr.reshape(-1, 1), yr)
        s_ps = lr.predict_proba(z.reshape(-1, 1))[:, 1]
        variants['перекалибровка: сдвиг и наклон'] = (s_ps, best_threshold(yr, s_ps[recent])[0])
        if M > T:
            s_re, thr_re, _ = fit_predict(win, FULL, y, known & ~recent, recent, SEED, X=X)
        else:
            s_re, thr_re = s_static, thr_static
        variants['дообучение'] = (s_re, thr_re)
        for v, (s, thr) in variants.items():
            pm = point_metrics(y[m_te], s[m_te], (s[m_te] > thr).astype(int))
            ratio, ece = calib(y[m_te], s[m_te])
            rows.append({'target': t, 'месяц': mo, 'стратегия': v, 'окон': int(m_te.sum()),
                         'доля событий': y[m_te].mean(), 'средний прогноз': s[m_te].mean(),
                         'прогноз / наблюдаемое': ratio, 'ECE': ece, 'AUROC': pm['AUROC'], 'AUPRC': pm['AUPRC'],
                         'macro_F1': pm['macro_F1'], 'sensitivity': pm['sensitivity'], 'PPV': pm['PPV'],
                         'тревог на пациента в неделю': (s[m_te] > thr).mean() * 7 / 3,
                         'окон для перекалибровки': int(recent.sum())})
            monthly_preds.setdefault((t, v), (np.full(len(win), np.nan), np.zeros(len(win), np.int8)))
            monthly_preds[(t, v)][0][m_te] = s[m_te]
            monthly_preds[(t, v)][1][m_te] = s[m_te] > thr
    print(f'{t}: {time.time() - t0:.0f} с')
shift = pd.DataFrame(rows)
shift.to_csv(os.path.join(OUT, 'temporal_strategies_by_month.csv'), index=False)
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 30)
display(shift.round(3))

In [ ]:
# Весь тестовый период: стратегии против статической модели (парный бутстреп по пациентам)
pooled = []
for t in S_TARGETS:
    y = win[f'y_{t}'].values
    names = [v for (tt, v) in monthly_preds if tt == t]
    m_all = te & ~np.isnan(monthly_preds[(t, 'статическая')][0])
    ent = [(m_all, y, np.nan_to_num(monthly_preds[(t, v)][0]), monthly_preds[(t, v)][1]) for v in names]
    B = paired_boot(ent, pid)
    j0 = names.index('статическая')
    for j, v in enumerate(names):
        s, p = monthly_preds[(t, v)]
        pm = point_metrics(y[m_all], s[m_all], p[m_all])
        ratio, ece = calib(y[m_all], s[m_all])
        r = {'target': t, 'стратегия': v, 'AUPRC': pm['AUPRC'], 'AUPRC 95% ДИ': ci(B[:, j, 1]),
             'macro_F1': pm['macro_F1'], 'ECE': ece, 'прогноз / наблюдаемое': ratio,
             'sensitivity': pm['sensitivity'], 'PPV': pm['PPV']}
        if j != j0:
            for mi, m in [(1, 'AUPRC'), (2, 'macro_F1')]:
                r[f'Δ{m} к статической 95% ДИ'] = ci(B[:, j, mi] - B[:, j0, mi])
        pooled.append(r)
pooled = pd.DataFrame(pooled)
pooled.to_csv(os.path.join(OUT, 'temporal_strategies_pooled.csv'), index=False)
display(pooled.round(3))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for col, t in enumerate(S_TARGETS):
    sub = shift[shift.target == t]
    for v, g in sub.groupby('стратегия', sort=False):
        axes[0, col].plot(g['месяц'], g['AUPRC'], marker='o', label=v)
        axes[1, col].plot(g['месяц'], g['прогноз / наблюдаемое'], marker='o', label=v)
    axes[0, col].set_title(f'{t}: AUPRC по месяцам')
    axes[1, col].axhline(1, color='k', lw=0.8, ls=':')
    axes[1, col].set_title(f'{t}: средний прогноз / наблюдаемая доля')
    axes[0, col].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'temporal_strategies.png'), dpi=200, bbox_inches='tight')
plt.show()

## 2. Качество по подгруппам

Предсказания XGB_full out-of-fold: из `oof_predictions.parquet` (если `clusterbp_v2.ipynb` уже посчитан), иначе модель
переобучается на тех же фолдах. Фенотип окна — по шаблонам его фолда.
`разброс` — разность максимума и минимума по группам признака (чувствительность и доля ложных тревог, FPR) —
простая мера равенства возможностей (equal opportunity).

In [ ]:
oof = load_oof_scores(win)
cl_oof = np.full(len(win), -1)
oof_scores = {}
for k in range(N_FOLDS):
    tr, va, te_k = fold_masks(win, k)
    cl_k = phenotypes(win, pheno_win, pheno_first, pheno_first_pid, k)
    cl_oof[te_k] = cl_k[te_k]
    for t in S_TARGETS:
        if oof is not None and f'{t}__XGB_full__score' in oof:
            continue
        y = win[f'y_{t}'].values
        s, thr, _ = fit_predict(win, FULL, y, tr, va, SEED + k)
        sc, pr = oof_scores.setdefault(t, (np.full(len(win), np.nan), np.zeros(len(win), np.int8)))
        sc[te_k], pr[te_k] = s[te_k], s[te_k] > thr
for t in S_TARGETS:
    if t not in oof_scores:
        oof_scores[t] = (oof[f'{t}__XGB_full__score'].values, oof[f'{t}__XGB_full__pred'].values)

age = win['age'].values
bmi = win['bmi'].values
arm = np.where(win['grp_experience'].values == 1, 'experience (реальная практика)',
               np.where(win['grp_control'].values == 1, 'control (контролируемая)', 'не указана'))
try:
    regions = load_regions()
    region = regions.reindex(pid).fillna('не указан').values
except Exception as e:      # справочник недоступен
    print('Регионы не загружены:', e)
    region = np.full(len(win), 'не указан')
nm_q = np.quantile(win['n_meas'].values, [1 / 3, 2 / 3])
GROUPS = {
    'возраст': pd.cut(age, [0, 50, 60, 70, 200], right=False, labels=['< 50', '50–59', '60–69', '≥ 70']).astype(str),
    'ИМТ': np.where(np.isnan(bmi), 'нет данных',
                    pd.cut(bmi, [0, 25, 30, 35, 100], right=False, labels=['< 25', '25–29.9', '30–34.9', '≥ 35']).astype(str)),
    'ветвь программы': arm,
    'измерений за 14 дней': np.where(win['n_meas'].values <= nm_q[0], f'≤ {nm_q[0]:.0f}',
                                     np.where(win['n_meas'].values <= nm_q[1], f'{nm_q[0]:.0f}–{nm_q[1]:.0f}',
                                              f'> {nm_q[1]:.0f}')),
    'дней в программе': pd.cut(win['days_in_program'].values, [0, 30, 90, 180, 10000], right=False,
                               labels=['14–29', '30–89', '90–179', '≥ 180']).astype(str),
    'фенотип': np.where(cl_oof < 0, 'не определён', np.char.add('кластер ', cl_oof.astype(str))),
    'регион': region,
}

rows, gaps = [], []
for t in S_TARGETS:
    y = win[f'y_{t}'].values
    s, p = oof_scores[t]
    for attr, g in GROUPS.items():
        levels = [lv for lv in sorted(pd.unique(g), key=str) if (g == lv).sum() >= 200 and y[g == lv].sum() >= 20]
        ent = [(g == lv, y, s, p) for lv in levels]
        B = paired_boot(ent, pid) if ent else None
        sens, fpr = [], []
        for j, lv in enumerate(levels):
            m = g == lv
            pm = point_metrics(y[m], s[m], p[m])
            ratio, ece = calib(y[m], s[m])
            rows.append({'target': t, 'признак': attr, 'группа': lv, 'пациентов': len(np.unique(pid[m])),
                         'окон': int(m.sum()), 'доля событий': pm['доля событий'], 'AUROC': pm['AUROC'],
                         'AUROC 95% ДИ': ci(B[:, j, 0]), 'AUPRC': pm['AUPRC'], 'AUPRC 95% ДИ': ci(B[:, j, 1]),
                         'AUPRC / доля': pm['AUPRC / доля'], 'sensitivity': pm['sensitivity'],
                         'FPR': 1 - pm['specificity'], 'PPV': pm['PPV'], 'прогноз / наблюдаемое': ratio, 'ECE': ece,
                         'тревог на пациента в неделю': p[m].mean() * 7 / 3})
            sens.append(pm['sensitivity'])
            fpr.append(1 - pm['specificity'])
        if len(levels) > 1:
            gaps.append({'target': t, 'признак': attr, 'групп': len(levels),
                         'разброс чувствительности': max(sens) - min(sens), 'разброс FPR': max(fpr) - min(fpr),
                         'мин. AUROC': min(r['AUROC'] for r in rows[-len(levels):]),
                         'макс. AUROC': max(r['AUROC'] for r in rows[-len(levels):])})
sub = pd.DataFrame(rows)
gap_df = pd.DataFrame(gaps)
sub.to_csv(os.path.join(OUT, 'subgroups.csv'), index=False)
gap_df.to_csv(os.path.join(OUT, 'subgroups_gaps.csv'), index=False)
display(sub[sub['признак'] != 'регион'].round(3))
display(gap_df.round(3))

In [ ]:
attrs = [a for a in GROUPS if a != 'регион']
fig, axes = plt.subplots(len(S_TARGETS), len(attrs), figsize=(3.2 * len(attrs), 3.4 * len(S_TARGETS)), squeeze=False)
for i, t in enumerate(S_TARGETS):
    for j, attr in enumerate(attrs):
        d = sub[(sub.target == t) & (sub['признак'] == attr)]
        ax = axes[i, j]
        x = np.arange(len(d))
        ax.bar(x - 0.2, d['sensitivity'], 0.4, label='чувствительность')
        ax.bar(x + 0.2, d['FPR'], 0.4, label='FPR')
        ax.set_xticks(x)
        ax.set_xticklabels(d['группа'], rotation=45, ha='right', fontsize=7)
        ax.set_title(f'{t}: {attr}', fontsize=9)
        if i == 0 and j == 0:
            ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(OUT, 'subgroups.png'), dpi=200, bbox_inches='tight')
plt.show()